# Step 3. Create a custom retriever

In [1]:
# RAG-ретривер на базе TF-IDF (Term Frequency-Inverse Document Frequency) можно назвать частотным ретривером, 
# так как его работа напрямую основана на статистике встречаемости слов.

In [2]:
from __future__ import annotations
import os
import pickle
from pathlib import Path
from typing import Any, Dict, Iterable, List, Tuple, Optional

from langchain_core.callbacks import CallbackManagerForRetrieverRun
from langchain_core.documents import Document
from langchain_core.retrievers import BaseRetriever

from gensim import corpora
from gensim.parsing import strip_tags, strip_numeric, \
    strip_multiple_whitespaces, stem_text, strip_punctuation, \
    remove_stopwords, preprocess_string, strip_non_alphanum
from gensim import models
from gensim import similarities
import pymorphy3

from retriever.stop_words import STOP_WORDS

In [3]:
morph_analyzer = pymorphy3.MorphAnalyzer()

In [4]:
SPLITS_PATH = './splits'
GENSIM_PATH = './gensim'

### Load TFIDF model files and splits

In [5]:
try:
    with open(os.path.join(GENSIM_PATH, 'dictionary.pkl'), 'rb') as h:
        dictionary = pickle.load(h)
    with open(os.path.join(GENSIM_PATH, 'similarity_index.pkl'), 'rb') as h:
        similarity_index = pickle.load(h)
    model_ = models.TfidfModel()
    retriever_model = model_.load(os.path.join(GENSIM_PATH, 'tfidf_model.mo'))
except FileNotFoundError:
    raise FileNotFoundError("Could not load gensim model files, please check gensim folder")

In [6]:
with open(os.path.join(SPLITS_PATH, 'splits.pkl'), 'rb') as f:
    splits = pickle.load(f)

### Query clening pipe

In [7]:
# Filters to be executed in pipeline
transform_to_lower = lambda s: s.lower()
CLEAN_FILTERS = [
                strip_tags,
                # strip_numeric,
                strip_punctuation,
                strip_non_alphanum,
                strip_multiple_whitespaces,
                transform_to_lower,
                ]

In [8]:
# Method does the filtering of all the unrelevant text elements
def cleaning_pipe(text:str) -> list[str]:
    # Invoking gensim.parsing.preprocess_string method with set of filters
    processed_words = preprocess_string(text, CLEAN_FILTERS)
    processed_words = [s for s in processed_words if len(s) > 1]
    processed_words = [s for s in processed_words if s not in STOP_WORDS]
    processed_words = [morph_analyzer.parse(s)[0].normal_form for s in processed_words]
    return processed_words

### Query engine for the custom retriever

In [9]:
def get_top_n(docs: List[Document], query: str, n: int, with_similarity: bool) -> List[Tuple[Document, float]] | List[Document] | None:
    """Retriever query engine
    Args:
        query: text query.
        n: number of returned documents.
    Returns:
        A list of tuples with relevance score.
    """
    query_bow = dictionary.doc2bow(query)
    sims = similarity_index[retriever_model[query_bow]]
    qty = sum(sims > 0)

    if qty > 0:
        top_idx = sims.argsort()[-1 * n:][::-1]
        result = []
        for idx in top_idx:
            similarity = round(float(sims[idx]), 3)
            doc = docs[idx]
            if with_similarity:
                result.append((doc, similarity))
            else:
                result.append(doc)
        return result
    else:
        return None

### Retriever template

In [10]:
class RetrieverTemplate(BaseRetriever):
    """Retriever template"""

    docs: List[Document]
    """Documents."""
    k: int = 3
    """Number of documents to return."""

    @classmethod
    def from_documents(
        cls,
        docs: Iterable[Document],
        **kwargs: Any,
    ) -> FreqRetriever:
        """
        Create a FreqRetriever instance from a list of langchain Documents.
        Args:
            docs: A list of of langchain Documents.
            **kwargs: Any other arguments to pass to the retriever.
        Returns:
            A Retriever instance.
        """
        return cls(
            docs=docs,
            **kwargs
        )

    def _get_relevant_documents(
        self, query: str, *, run_manager: CallbackManagerForRetrieverRun
    ) -> List[Tuple[Document, float]] | None:
        
        return_docs = self.docs[:self.k]
        return return_docs

In [11]:
retriever = RetrieverTemplate.from_documents(
    docs=splits,
    k = 3,
)

result = retriever.invoke('')
for doc in result:
    print(doc.metadata)

{'title': 'Уильям Гибсон. Виртуальный свет', 'chapter': 'ОГЛАВЛЕНИЕ', 'id': '02cb059b44a9e371accff78680bf949d', 'size': 883, 'collection': 'Virtual_Light'}
{'title': 'Уильям Гибсон. Виртуальный свет', 'chapter': '1. ПЛАМЕНЕЮЩАЯ ПЛОТЬ ИСПОЛИНОВ', 'section': 'Курьер прислонился лбом к...', 'id': '10d86dbb6e6d10ebaf27485597346360', 'size': 2102, 'collection': 'Virtual_Light'}
{'title': 'Уильям Гибсон. Виртуальный свет', 'chapter': '1. ПЛАМЕНЕЮЩАЯ ПЛОТЬ ИСПОЛИНОВ', 'section': 'Чуть заполночь на...', 'id': '9359fc2a6ceefb8fcf1c835c99e362e2', 'size': 4358, 'collection': 'Virtual_Light'}


### Custom retriever class

In [12]:
class FreqRetriever(BaseRetriever):
    """TF-IDF retriever based on gensim model"""

    docs: List[Document]
    """Documents."""
    k: int = 4
    """Number of documents to return."""
    with_similarity: bool = False
    """True for return found chunk relevance"""

    @classmethod
    def from_documents(
        cls,
        docs: Iterable[Document],
        **kwargs: Any,
    ) -> FreqRetriever:
        """
        Create a FreqRetriever instance from a list of langchain Documents.
        Args:
            docs: A list of of langchain Documents.
            **kwargs: Any other arguments to pass to the retriever.
        Returns:
            A FreqRetriever instance.
        """
        return cls(
            docs=docs,
            **kwargs
        )

    def _get_relevant_documents(
        self, query: str, *, run_manager: CallbackManagerForRetrieverRun
    ) -> List[Tuple[Document, float]] | None:
        processed_query = cleaning_pipe(query)
        return_docs = get_top_n(self.docs, processed_query, n=self.k, with_similarity=self.with_similarity)
        return return_docs

### Custom retriever test

In [13]:
retriever = FreqRetriever.from_documents(
    docs=splits,
    k = 3,
    with_similarity = True
)

In [14]:
def retriever_test(query: str) -> None:
    result = retriever.invoke(query)
    if not result:
        print("No relevant documents found")
        return None
    if hasattr(retriever, 'with_similarity') and retriever.with_similarity:
        for doc, similarity in result:
            print(f'Similarity: {similarity}')
            print(doc.metadata)
            print(f'{doc.page_content[:500]}...\n')
    else:
        for doc in result:
            print(doc.metadata)
            print(f'{doc.page_content[:500]}...\n')            

In [15]:
query1 = """cyberpunk"""
query2 = """Какой персонаж, работающий на 'Интенсекьюр' уже два года, рассказал Райделлу о существовании домов-невидимок, известных как 'стелсы'?"""

In [16]:
retriever_test(query1)

No relevant documents found


In [17]:
retriever_test(query2)

Similarity: 0.236
{'title': 'Уильям Гибсон. Виртуальный свет', 'chapter': '2. "ГРОМИЛА" В ДЕЛЕ', 'section': 'Райделл вставил ключ...', 'id': '6851dc51ddcc613e44670dfc2c6ec6f3', 'size': 8156, 'collection': 'Virtual_Light'}
2. "ГРОМИЛА" В ДЕЛЕ

Райделл вставил ключ, набрал на клавиатуре пароль, прогнал через систему тестовую программу, и серые прямоугольники дисплеев ожили. Ему очень нравились камеры, установленные под задним бампером "Громилы": это ж какое удобство при парковке, когда подаешь машину задом и точно видишь - куда.
Связь со "Звездой Смерти" пока что отсутствовала, слишком уж много металла в стенах и перекрытиях автомойки. Ничего, у Саблетта в ухе капсульный приемник, вот пусть и слушает, все равно ему...

Similarity: 0.039
{'title': 'Уильям Гибсон. Виртуальный свет', 'chapter': '26. "ЦВЕТНЫЕ ЛЮДИ"', 'section': 'Может в этом мормонском...', 'id': '34f55a652d6210d2b3c02207b81a6961', 'size': 11731, 'collection': 'Virtual_Light'}
26. "ЦВЕТНЫЕ ЛЮДИ"

Может, в этом мормонском чае